# Primary Import and Export Countries by State

Following the announcement by the new administration to raise tariffs on Canada, Mexico, and China (and the not entirely unexpected retaliatory measures from those targeted countries), it would be interesting to know which countries are the primary sources and destinations for imports and exports.

This data can be viewed using the US government’s International Trade Administration the page located at [https://www.trade.gov/data-visualization/tradestats-express-state-trade-product](https://www.trade.gov/data-visualization/tradestats-express-state-trade-product) although that page is simply a container for n iframe whose contents are actually located at [https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyPartners?%3Atoolbar=yes%2Ctop&%3AshowVizHome=no&%3Aembed=true](https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyPartners?%3Atoolbar=yes%2Ctop&%3AshowVizHome=no&%3Aembed=true) and are themselves created using Tableau.

However, scraping this page is problematic, not least becuase of all the DOM selector work that needs to be done, as well as interacting with the page once it loads (to click on combo box items, etc.), but also because the embedded link contained within the iframe changes constantly (by the time you check again, this link may no longer be valid, even though as of this writing, it is working).

Instead, we can go to the source of this data which is actually hosted on the US government's census.gov site and source the required data using the available API.

After acquiring the data, we also want to display it graphically using a map of the USA to show each state’s dependence on their primary export and import partners. For this, we will employ Choropleth maps.

In [ ]:
# Import the libraries we will use to scrape data
from dotenv import load_dotenv
import os
from rich import print_json
from rich.console import Console
from rich.theme import Theme

console = Console(theme=Theme({
    "json.key":    "bold #61afef",   # light blue
    "json.str":    "#98c379",        # green
    "json.number": "bold #e5c07b",   # yellow
    "json.brace":  "bold #abb2bf",   # light grey
    "json.null":   "italic #c678dd",
    "json.bool_true":  "italic #98c379",
    "json.bool_false": "italic #e06c75",
}))

from datetime import datetime, timedelta, timezone
import pandas as pd

# For plotting USA map
import plotly.express as px

In [2]:
# Some configurations required for the remainder of the script

load_dotenv(override=True)
CENSUS_GOV_API_KEY = os.getenv("CENSUS_GOV_API_KEY")
recent_full_year = datetime.now().year - 1
census_exports_url = f"https://api.census.gov/data/timeseries/intltrade/exports/statehs?get=STATE,CTY_CODE,CTY_NAME,ALL_VAL_YR&time={recent_full_year}-12&STATE=[STATE]&key={CENSUS_GOV_API_KEY}"
census_imports_url = f"https://api.census.gov/data/timeseries/intltrade/imports/statehs?get=STATE,CTY_CODE,CTY_NAME,GEN_VAL_YR&time={recent_full_year}-12&STATE=[STATE]&key={CENSUS_GOV_API_KEY}"

In [3]:
# We also need a translation dictionary from state name to state abbreviation (for mapping)

dic_states = {
	  "Alabama": "AL"
	, "Alaska": "AK"
	, "Arizona": "AZ"
	, "Arkansas": "AR"
	, "California": "CA"
	, "Colorado": "CO"
	, "Connecticut": "CT"
	, "Delaware": "DE"
	, "District of Columbia": "DC"
	, "Florida": "FL"
	, "Georgia": "GA"
	, "Hawaii": "HI"
	, "Idaho": "ID"
	, "Illinois": "IL"
	, "Indiana": "IN"
	, "Iowa": "IA"
	, "Kansas": "KS"
	, "Kentucky": "KY"
	, "Louisiana": "LA"
	, "Maine": "ME"
	, "Maryland": "MD"
	, "Massachusetts": "MA"
	, "Michigan": "MI"
	, "Minnesota": "MN"
	, "Mississippi": "MS"
	, "Missouri": "MO"
	, "Montana": "MT"
	, "Nebraska": "NE"
	, "Nevada": "NV"
	, "New Hampshire": "NH"
	, "New Jersey": "NJ"
	, "New Mexico": "NM"
	, "New York": "NY"
	, "North Carolina": "NC"
	, "North Dakota": "ND"
	, "Ohio": "OH"
	, "Oklahoma": "OK"
	, "Oregon": "OR"
	, "Pennsylvania": "PA"
	, "Puerto Rico": "PR"
	, "Rhode Island": "RI"
	, "South Carolina": "SC"
	, "South Dakota": "SD"
	, "Tennessee": "TN"
	, "Texas": "TX"
	, "Utah": "UT"
	, "Vermont": "VT"
	, "Virginia": "VA"
	, "Washington": "WA"
	, "West Virginia": "WV"
	, "Wisconsin": "WI"
	, "Wyoming": "WY"
}

In [4]:
# Set up our dictionaries for import countries by state
state_exports = {}
state_exports_canada = {} #Specifically to report on Canada exports

for state_name, state_abbr in dic_states.items():
    state_url = census_exports_url.replace("[STATE]", state_abbr)
    df_state = pd.read_csv(
          state_url
        , skiprows=1
        , usecols = [0, 1, 2, 3, 4]
        , names = ["STATE_ABBR", "COUNTRY_CODE", "COUNTRY_NAME", "VALUE", "PERIOD"]
        , converters = {0: lambda x: x.replace('"', '').replace("[", ""), 4: lambda x: pd.to_datetime(x, format='%Y-%m') + pd.offsets.MonthEnd(0)}
    )
    # Extract the world-wide total to calculate percent of trade by country
    df_total = df_state[df_state["COUNTRY_CODE"] == "-"]
    # Filter only for countries, not regional or other totals
    df_state = df_state[df_state["COUNTRY_CODE"].str.contains(r"^[1-9][0-9]+$")]
    # Create a new column showin gpercentage of trade for each country as a proportion of the total world trade for this state
    df_state["PERCENT"] = df_state["VALUE"] / df_total["VALUE"].iloc[0]
    # Sort by trade value in percentage
    df_state.sort_values("PERCENT", ascending=False, inplace=True)

    # After sorting, make a copy containing only Canada imports
    df_state_canada = df_state.copy()
    df_state_canada = df_state_canada[df_state_canada["COUNTRY_CODE"] == "1220"]
    # Keep only the row for Canada
    df_state_canada = df_state_canada.reset_index(drop=True).head(1)

    # Highest valued import country for this state
    df_state = df_state.reset_index(drop=True).head(1)
    # Print just one of the states' export data
    if state_abbr == "FL":
        console.print_json(data=df_state.to_dict(orient="records"), ensure_ascii=True, default=str, highlight=True, indent=4)
    # Add the highest valued export country for this state to the state_exports dictionary
    state_exports[state_name] = df_state.iloc[0].to_dict()

    state_exports_canada[state_name] = df_state_canada.iloc[0].to_dict()

[
    {
        "STATE_ABBR": "FL",
        "COUNTRY_CODE": "3510",
        "COUNTRY_NAME": "BRAZIL",
        "VALUE": 6059499759,
        "PERIOD": "2025-12-31 00:00:00",
        "PERCENT": 0.07660292298649299
    }
]

In [5]:
# Set up our dictionaries for import countries by state
state_imports = {}
state_imports_canada = {} #Specifically to report on Canada imports

for state_name, state_abbr in dic_states.items():
    state_url = census_imports_url.replace("[STATE]", state_abbr)
    df_state = pd.read_csv(
          state_url
        , skiprows=1
        , usecols = [0, 1, 2, 3, 4]
        , names = ["STATE_ABBR", "COUNTRY_CODE", "COUNTRY_NAME", "VALUE", "PERIOD"]
        , converters = {0: lambda x: x.replace('"', '').replace("[", ""), 4: lambda x: pd.to_datetime(x, format='%Y-%m') + pd.offsets.MonthEnd(0)}
    )
    # Extract the world-wide total to calculate percent of trade by country
    df_total = df_state[df_state["COUNTRY_CODE"] == "-"]
    # Filter only for countries, not regional or other totals
    df_state = df_state[df_state["COUNTRY_CODE"].str.contains(r"^[1-9][0-9]+$")]
    # Create a new column showin gpercentage of trade for each country as a proportion of the total world trade for this state
    df_state["PERCENT"] = df_state["VALUE"] / df_total["VALUE"].iloc[0]
    # Sort by trade value in percentage
    df_state.sort_values("PERCENT", ascending=False, inplace=True)

    # After sorting, make a copy containing only Canada imports
    df_state_canada = df_state.copy()
    df_state_canada = df_state_canada[df_state_canada["COUNTRY_CODE"] == "1220"]
    # Keep only the row for Canada
    df_state_canada = df_state_canada.reset_index(drop=True).head(1)

    # Highest valued import country for this state
    df_state = df_state.reset_index(drop=True).head(1)
    # Print just one of the states' export data

    if state_abbr == "FL":
        console.print_json(data=df_state.to_dict(orient="records"), ensure_ascii=True, default=str, highlight=True, indent=4)
    # Add the highest valued import country for this state to the state_imports dictionary
    state_imports[state_name] = df_state.iloc[0].to_dict()

    state_imports_canada[state_name] = df_state_canada.iloc[0].to_dict()

[
    {
        "STATE_ABBR": "FL",
        "COUNTRY_CODE": "5700",
        "COUNTRY_NAME": "CHINA",
        "VALUE": 9376055460,
        "PERIOD": "2025-12-31 00:00:00",
        "PERCENT": 0.08091909111534877
    }
]

In [6]:
# Let's look at the first 10 rows of data (in JSON format) we collected for export countries

console.print_json(data=list(state_exports.items())[:10], ensure_ascii=True, default=str, highlight=True, indent=4)

[
    [
        "Alabama",
        {
            "STATE_ABBR": "AL",
            "COUNTRY_CODE": "1220",
            "COUNTRY_NAME": "CANADA",
            "VALUE": 4308595257,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.1820204491224145
        }
    ],
    [
        "Alaska",
        {
            "STATE_ABBR": "AK",
            "COUNTRY_CODE": "5800",
            "COUNTRY_NAME": "KOREA, SOUTH",
            "VALUE": 1092750433,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.16315359914900215
        }
    ],
    [
        "Arizona",
        {
            "STATE_ABBR": "AZ",
            "COUNTRY_CODE": "2010",
            "COUNTRY_NAME": "MEXICO",
            "VALUE": 14582642604,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.3266982923012144
        }
    ],
    [
        "Arkansas",
        {
            "STATE_ABBR": "AR",
            "COUNTRY_CODE": "2010",
            "COUNTRY_NAME": "MEXICO",
            "VALUE": 1495637927,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.22471479556214624
        }
    ],
    [
        "California",
        {
            "STATE_ABBR": "CA",
            "COUNTRY_CODE": "2010",
            "COUNTRY_NAME": "MEXICO",
            "VALUE": 34868636147,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.18531435771826138
        }
    ],
    [
        "Colorado",
        {
            "STATE_ABBR": "CO",
            "COUNTRY_CODE": "2010",
            "COUNTRY_NAME": "MEXICO",
            "VALUE": 1706496654,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.14825735153769945
        }
    ],
    [
        "Connecticut",
        {
            "STATE_ABBR": "CT",
            "COUNTRY_CODE": "1220",
            "COUNTRY_NAME": "CANADA",
            "VALUE": 2199733067,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.12356346791809152
        }
    ],
    [
        "Delaware",
        {
            "STATE_ABBR": "DE",
            "COUNTRY_CODE": "1220",
            "COUNTRY_NAME": "CANADA",
            "VALUE": 774832037,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.14068897971616762
        }
    ],
    [
        "District of Columbia",
        {
            "STATE_ABBR": "DC",
            "COUNTRY_CODE": "5170",
            "COUNTRY_NAME": "SAUDI ARABIA",
            "VALUE": 1494385078,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.40868199149427087
        }
    ],
    [
        "Florida",
        {
            "STATE_ABBR": "FL",
            "COUNTRY_CODE": "3510",
            "COUNTRY_NAME": "BRAZIL",
            "VALUE": 6059499759,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.07660292298649299
        }
    ]
]

In [7]:
# Let's look at the first 10 rows of data (in JSON format) we collected for import countries

console.print_json(data=list(state_imports.items())[:10], ensure_ascii=True, default=str, highlight=True, indent=4)

[
    [
        "Alabama",
        {
            "STATE_ABBR": "AL",
            "COUNTRY_CODE": "2010",
            "COUNTRY_NAME": "MEXICO",
            "VALUE": 6506771200,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.18614800002495002
        }
    ],
    [
        "Alaska",
        {
            "STATE_ABBR": "AK",
            "COUNTRY_CODE": "1220",
            "COUNTRY_NAME": "CANADA",
            "VALUE": 1278531519,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.4070541937080744
        }
    ],
    [
        "Arizona",
        {
            "STATE_ABBR": "AZ",
            "COUNTRY_CODE": "5830",
            "COUNTRY_NAME": "TAIWAN",
            "VALUE": 16686681017,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.29121246040573384
        }
    ],
    [
        "Arkansas",
        {
            "STATE_ABBR": "AR",
            "COUNTRY_CODE": "1220",
            "COUNTRY_NAME": "CANADA",
            "VALUE": 1064656428,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.15956240838201466
        }
    ],
    [
        "California",
        {
            "STATE_ABBR": "CA",
            "COUNTRY_CODE": "5700",
            "COUNTRY_NAME": "CHINA",
            "VALUE": 76083138784,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.15588795042194387
        }
    ],
    [
        "Colorado",
        {
            "STATE_ABBR": "CO",
            "COUNTRY_CODE": "1220",
            "COUNTRY_NAME": "CANADA",
            "VALUE": 5321615753,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.325008648071228
        }
    ],
    [
        "Connecticut",
        {
            "STATE_ABBR": "CT",
            "COUNTRY_CODE": "1220",
            "COUNTRY_NAME": "CANADA",
            "VALUE": 5716766651,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.23793677631352414
        }
    ],
    [
        "Delaware",
        {
            "STATE_ABBR": "DE",
            "COUNTRY_CODE": "2010",
            "COUNTRY_NAME": "MEXICO",
            "VALUE": 988128440,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.09528098013818033
        }
    ],
    [
        "District of Columbia",
        {
            "STATE_ABBR": "DC",
            "COUNTRY_CODE": "6021",
            "COUNTRY_NAME": "AUSTRALIA",
            "VALUE": 341461477,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.3172498267936416
        }
    ],
    [
        "Florida",
        {
            "STATE_ABBR": "FL",
            "COUNTRY_CODE": "5700",
            "COUNTRY_NAME": "CHINA",
            "VALUE": 9376055460,
            "PERIOD": "2025-12-31 00:00:00",
            "PERCENT": 0.08091909111534877
        }
    ]
]

In [8]:
# Using the site:
# https://medialab.github.io/iwanthue/
# I generated a set of 30 colors by setting the Color Space attributes H, and C to their full range but keeping L between 50 and 100
# Cycle through the color by rerolling the palette and locking in those you like
# The resulting set is sorted by diff and we can use up to those 30 colors to assign to all the countries present in both import and export sets

country_colors = [
      "#60a386"
    , "#d74de4"
    , "#73ea43"
    , "#a175e4"
    , "#c9e341"
    , "#e260b5"
    , "#63c851"
    , "#e94d7b"
    , "#63e4a8"
    , "#ec5f33"
    , "#6988dd"
    , "#e1c13a"
    , "#77a8de"
    , "#d58c31"
    , "#54b0c4"
    , "#dc6969"
    , "#55a458"
    , "#c991c8"
    , "#b4d86d"
    , "#d38c96"
    , "#7ce1dc"
    , "#d08c62"
    , "#c1cfe2"
    , "#949639"
    , "#8592ab"
    , "#e2cd83"
    , "#9a968b"
    , "#b5dca9"
    , "#9e8f65"
    , "#e1ccbb"
]

# Now assign the first X of these colors to the number of unique countries:

unique_countries = sorted((pd.concat([pd.DataFrame(state_exports).T["COUNTRY_NAME"], pd.DataFrame(state_imports).T["COUNTRY_NAME"]]).unique().tolist()))

SHARED_COLOR_MAP = dict(zip(unique_countries, country_colors[:len(unique_countries)]))
SHARED_COLOR_MAP

{'AUSTRALIA': '#60a386',
 'BRAZIL': '#d74de4',
 'CANADA': '#73ea43',
 'CHINA': '#a175e4',
 'COSTA RICA': '#c9e341',
 'GERMANY': '#e260b5',
 'INDIA': '#63c851',
 'IRELAND': '#e94d7b',
 'ITALY': '#63e4a8',
 'JAPAN': '#ec5f33',
 'KOREA, SOUTH': '#6988dd',
 'MALAYSIA': '#e1c13a',
 'MEXICO': '#77a8de',
 'NETHERLANDS': '#d58c31',
 'SAUDI ARABIA': '#54b0c4',
 'SPAIN': '#dc6969',
 'SWITZERLAND': '#55a458',
 'TAIWAN': '#c991c8',
 'THAILAND': '#b4d86d',
 'UNITED KINGDOM': '#d38c96'}

In [9]:
df_exports = pd.DataFrame.from_dict(state_exports, columns=["STATE_ABBR", "COUNTRY_NAME", "VALUE", "PERCENT", "PERIOD"], orient="index")
df_exports.index.names = ["STATE"]
df_exports.reset_index(drop=False, inplace=True)
# Since Puerto Rico doesn't appear on the map, we will drop it from the dataframe
df_exports = df_exports[df_exports["STATE_ABBR"] != "PR"]
df_exports.head(10)

,STATE,STATE_ABBR,COUNTRY_NAME,VALUE,PERCENT,PERIOD
0,Alabama,AL,CANADA,4308595257,0.182020,2025-12-31
1,Alaska,AK,"KOREA, SOUTH",1092750433,0.163154,2025-12-31
2,Arizona,AZ,MEXICO,14582642604,0.326698,2025-12-31
3,Arkansas,AR,MEXICO,1495637927,0.224715,2025-12-31
4,California,CA,MEXICO,34868636147,0.185314,2025-12-31
5,Colorado,CO,MEXICO,1706496654,0.148257,2025-12-31
6,Connecticut,CT,CANADA,2199733067,0.123563,2025-12-31
7,Delaware,DE,CANADA,774832037,0.140689,2025-12-31
8,District of Columbia,DC,SAUDI ARABIA,1494385078,0.408682,2025-12-31
9,Florida,FL,BRAZIL,6059499759,0.076603,2025-12-31


In [10]:
# We will only label the states whose primary trading partner is Canada

df_export_labels = df_exports[df_exports["COUNTRY_NAME"] == "CANADA"]
df_export_labels = df_export_labels.drop(columns=["COUNTRY_NAME"])
df_export_labels["Label_Text"] = df_export_labels.apply(
      lambda row: "${:,.0f}<br>{:.2%}".format(row["VALUE"], row["PERCENT"])
    , axis=1
)
df_export_labels

,STATE,STATE_ABBR,VALUE,PERCENT,PERIOD,Label_Text
0,Alabama,AL,4308595257,0.182020,2025-12-31,"$4,308,595,257<br>18.20%"
6,Connecticut,CT,2199733067,0.123563,2025-12-31,"$2,199,733,067<br>12.36%"
7,Delaware,DE,774832037,0.140689,2025-12-31,"$774,832,037<br>14.07%"
10,Georgia,GA,6882308605,0.114154,2025-12-31,"$6,882,308,605<br>11.42%"
12,Idaho,ID,1780759590,0.386328,2025-12-31,"$1,780,759,590<br>38.63%"
13,Illinois,IL,18089216763,0.225565,2025-12-31,"$18,089,216,763<br>22.56%"
15,Iowa,IA,4957867389,0.303984,2025-12-31,"$4,957,867,389<br>30.40%"
17,Kentucky,KY,9426307842,0.182497,2025-12-31,"$9,426,307,842<br>18.25%"
19,Maine,ME,1316403008,0.412228,2025-12-31,"$1,316,403,008<br>41.22%"
20,Maryland,MD,2178793319,0.131558,2025-12-31,"$2,178,793,319<br>13.16%"


In [11]:
fig = px.choropleth(
      df_exports
    , locations = "STATE_ABBR"
    , locationmode = "USA-states"
    , color = "COUNTRY_NAME"
    , scope = "usa"
    , hover_data = {"STATE_ABBR": False, "STATE": True, "COUNTRY_NAME": True}
    #, labels = {"COUNTRY_NAME": "Target", "STATE_ABBR":""}
    , title = "Primary Export Country by State"
    #, width = 1024
    #, height = 768
    , category_orders = {"COUNTRY_NAME": sorted(pd.unique(df_exports["COUNTRY_NAME"]))}
    #, color_discrete_sequence = px.colors.qualitative.Alphabet[1:]
    #, color_discrete_sequence = px.colors.qualitative.Light24
    , color_discrete_map = SHARED_COLOR_MAP
)

fig.add_scattergeo(
      locations = df_export_labels["STATE_ABBR"]
    , locationmode = "USA-states"
    , text = df_export_labels["Label_Text"]
    , mode = "text"
    , textfont = dict(
          size = 10
        , color = "black"
        , family = "Arial-Bold"
    )
    , showlegend=False  # Hides the scatter trace from the legend
)

fig.update_layout(
      width = 1000
    , height = 500
    , margin = {"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()
fig.write_image("images/export_countries_by_state.png")

![export_countries_by_state.png](./images/export_countries_by_state.png)

In [12]:
df_imports = pd.DataFrame.from_dict(state_imports, columns=["STATE_ABBR", "COUNTRY_NAME", "VALUE", "PERCENT", "PERIOD"], orient="index")
df_imports.index.names = ["STATE"]
df_imports.reset_index(drop=False, inplace=True)
# Since Puerto Rico doesn't appear on the map, we will drop it from the dataframe
df_imports = df_imports[df_imports["STATE_ABBR"] != "PR"]
df_imports.head(10)

,STATE,STATE_ABBR,COUNTRY_NAME,VALUE,PERCENT,PERIOD
0,Alabama,AL,MEXICO,6506771200,0.186148,2025-12-31
1,Alaska,AK,CANADA,1278531519,0.407054,2025-12-31
2,Arizona,AZ,TAIWAN,16686681017,0.291212,2025-12-31
3,Arkansas,AR,CANADA,1064656428,0.159562,2025-12-31
4,California,CA,CHINA,76083138784,0.155888,2025-12-31
5,Colorado,CO,CANADA,5321615753,0.325009,2025-12-31
6,Connecticut,CT,CANADA,5716766651,0.237937,2025-12-31
7,Delaware,DE,MEXICO,988128440,0.095281,2025-12-31
8,District of Columbia,DC,AUSTRALIA,341461477,0.317250,2025-12-31
9,Florida,FL,CHINA,9376055460,0.080919,2025-12-31


In [13]:
# We will only label the states whose primary trading partner is Canada

df_import_labels = df_imports[df_imports["COUNTRY_NAME"] == "CANADA"]
df_import_labels = df_import_labels.drop(columns=["COUNTRY_NAME"])
df_import_labels["Label_Text"] = df_import_labels.apply(
      lambda row: "${:,.0f}<br>{:.2%}".format(row["VALUE"], row["PERCENT"])
    , axis=1
)
df_import_labels

,STATE,STATE_ABBR,VALUE,PERCENT,PERIOD,Label_Text
1,Alaska,AK,1278531519,0.407054,2025-12-31,"$1,278,531,519<br>40.71%"
3,Arkansas,AR,1064656428,0.159562,2025-12-31,"$1,064,656,428<br>15.96%"
5,Colorado,CO,5321615753,0.325009,2025-12-31,"$5,321,615,753<br>32.50%"
6,Connecticut,CT,5716766651,0.237937,2025-12-31,"$5,716,766,651<br>23.79%"
13,Illinois,IL,58308646722,0.276514,2025-12-31,"$58,308,646,722<br>27.65%"
15,Iowa,IA,3180022312,0.265599,2025-12-31,"$3,180,022,312<br>26.56%"
16,Kansas,KS,1973585769,0.141796,2025-12-31,"$1,973,585,769<br>14.18%"
19,Maine,ME,4118209841,0.686894,2025-12-31,"$4,118,209,841<br>68.69%"
21,Massachusetts,MA,9964422432,0.232596,2025-12-31,"$9,964,422,432<br>23.26%"
23,Minnesota,MN,12501156566,0.335502,2025-12-31,"$12,501,156,566<br>33.55%"


In [14]:
fig = px.choropleth(
      df_imports
    , locations = "STATE_ABBR"
    , locationmode = "USA-states"
    , color = "COUNTRY_NAME"
    , scope = "usa"
    , hover_data = {"STATE_ABBR": False, "STATE": True, "COUNTRY_NAME": True}
    #, labels = {"COUNTRY_NAME": "Target", "STATE_ABBR":""}
    , title = "Primary Import Country by State"
    #, width = 1024
    #, height = 768
    , category_orders = {"COUNTRY_NAME": sorted(pd.unique(df_imports["COUNTRY_NAME"]))}
    #, color_discrete_sequence = px.colors.qualitative.Alphabet[1:]
    #, color_discrete_sequence = px.colors.qualitative.Light24
    , color_discrete_map = SHARED_COLOR_MAP
)

fig.add_scattergeo(
      locations = df_import_labels["STATE_ABBR"]
    , locationmode = "USA-states"
    , text = df_import_labels["Label_Text"]
    , mode = "text"
    , textfont = dict(
          size = 10
        , color = "black"
        , family = "Arial-Bold"
    )
    , showlegend=False  # Hides the scatter trace from the legend
)

fig.update_layout(
      width = 1000
    , height = 500
    , margin = {"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()
fig.write_image("images/import_countries_by_state.png")

![import_countries_by_state.png](./images/import_countries_by_state.png)

In [15]:
# As a final step, generate tables for the export and import each state has specifically just with Canada

df_exports_canada = df_exports[df_exports["COUNTRY_NAME"] == "CANADA"].sort_values("PERCENT", ascending=False)[["STATE", "PERCENT"]]
avg_canada_exports = df_exports["PERCENT"].mean()
df_exports_canada = df_exports_canada[df_exports_canada["PERCENT"] > avg_canada_exports]
df_exports_canada["PERCENT"] = df_exports_canada.apply(lambda row: "{:.2%}".format(row["PERCENT"]), axis=1)

str_export_canada = f"""
Following the announcement that Canada will suffer tariffs of 50% on exports to the USA,
and the reciprocal tariffs enacted by Canada on US exports, based on the stats collected
from the trade.gov website show that the following states all have the percent of goods
and exports to Canada in proportion to the exports to the entire world exceeding
the national average of {avg_canada_exports:.2%}. The reciprocal tariffs are going to bite hard on these states.
"""

print(str_export_canada)
df_exports_canada.set_index("STATE")


Following the announcement that Canada will suffer tariffs of 50% on exports to the USA,
and the reciprocal tariffs enacted by Canada on US exports, based on the stats collected
from the trade.gov website show that the following states all have the percent of goods
and exports to Canada in proportion to the exports to the entire world exceeding
the national average of 25.80%. The reciprocal tariffs are going to bite hard on these states.



,PERCENT
STATE,
North Dakota,80.60%
Montana,46.55%
South Dakota,45.87%
West Virginia,44.10%
Maine,41.22%
Idaho,38.63%
Michigan,38.53%
Missouri,34.14%
Ohio,32.34%


In [16]:
df_imports_canada = df_imports[df_exports["COUNTRY_NAME"] == "CANADA"].sort_values("PERCENT", ascending=False)[["STATE", "PERCENT"]]
avg_canada_imports = df_imports["PERCENT"].mean()
df_imports_canada = df_imports_canada[df_imports_canada["PERCENT"] > avg_canada_imports]
df_imports_canada["PERCENT"] = df_imports_canada.apply(lambda row: "{:.2%}".format(row["PERCENT"]), axis=1)

str_import_canada = f"""
Similarly, the states with the percentage of their imports from Canada in proportion to goods and services
imported from the rest of the world exceeding the average across the USA of {avg_canada_imports:.2%} are:
"""

print(str_import_canada)
df_imports_canada.set_index("STATE")


Similarly, the states with the percentage of their imports from Canada in proportion to goods and services
imported from the rest of the world exceeding the average across the USA of 28.39% are:



,PERCENT
STATE,
Montana,91.37%
Vermont,70.02%
Maine,68.69%
North Dakota,67.57%
Wyoming,48.68%
Michigan,47.24%
Oklahoma,45.91%
West Virginia,37.73%
South Dakota,36.01%
